# __Inteligencia Artificial - IUA 2026__

### *Pipeline EDA MOD1: Limpieza y preparacion de los datos para entrenamiento*

### Integrantes
* **Celeste Antonella Badami** — cbadami845@alumnos.iua.edu.ar
* **Benjamin Vargas** — bvargas161@alumnos.iua.edu.ar

---


## 1. Descripcion del modelo Pipeline (TODO)
Transformaciones:
* 
Eliminaciones:
* Este modelo de pipeline descarta los viajes en los cuales se detecto activa la flag de desvio.


## 2. Preparación del entorno

In [155]:
# importación de librerias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [156]:
# Carga de la muestra de datos preeliminar
df = pd.read_csv('../data/dataset.csv')

### 2.1 Métricas del modelo de pipeline

In [ ]:
numero_filas_iniciales = len(df)
numero_columnas_iniciales = len(df.columns)

## 3. Pipeline EDA

### 3.1 Transformación de datos

In [158]:
# Convertir a datetime (utc=True maneja automáticamente el +00)
df['date_time'] = pd.to_datetime(df['date_time'], utc=True)

#### 3.1.2 Implementación de Hora cíclica

In [159]:
segundos_del_dia = (
    df['date_time'].dt.hour * 3600 + 
    df['date_time'].dt.minute * 60 + 
    df['date_time'].dt.second
)

SEGUNDOS_EN_DIA = 24 * 3600  # 86400

# Aplicar Seno y Coseno
df['hora_sin'] = np.sin(2 * np.pi * segundos_del_dia / SEGUNDOS_EN_DIA)
df['hora_cos'] = np.cos(2 * np.pi * segundos_del_dia / SEGUNDOS_EN_DIA)

#### 3.1.3 Label Encoding sobre estados del colectivo

In [160]:
df['current_status_encoded'] = df['current_status'].map({
    'STOPPED_AT': 0, 
    'IN_TRANSIT_TO': 1
})

#### 3.1.3 Codificación de día de la semana

In [161]:
# Extraer día de la semana (0 = Lunes, 6 = Domingo)
df['dia_semana'] = df['date_time'].dt.day_of_week

#### 3.1.3 Ordinal encoding sobre estado de ocupación

In [162]:
mapeo_ocupacion = {
    'MANY_SEATS_AVAILABLE': 1,
    'FEW_SEATS_AVAILABLE': 2,
    'FULL': 3
}

# Realiza el mapeo manteniendo los NaN
df['occupancy_encoded'] = df['occupancy_status'].map(mapeo_ocupacion).fillna(0).astype(int)

> ⚠️ **ADVERTENCIA:** Este bloque realiza el mapeo de categorías pero **deja pasar los valores `NaN` intactos** en `occupancy_encoded` asignandoles un valor 0. Asegúrarse de tratarlos

#### 3.1.4 Mapeo binario para saber si es de día

In [163]:
df['is_day_encoded'] = df['is_day'].astype('Int64')

> ⚠️ **ADVERTENCIA:** Este bloque realiza el mapeo de categorías pero **deja pasar los valores `NaN` intactos** en `is_day_encoded` (los cuales pandas representará como `NaN` de tipo `float64` o `<NA>` de tipo `Int64`). Asegúrarse de tratarlos

### 3.2 Generación de nuevas features

In [164]:
# Crear una variable booleana / binaria para fin de semana
df['es_fin_de_semana'] = df['dia_semana'].isin([5, 6]).astype(int)

### 3.2 Agrupación de datos (Conformación de viajes)

In [165]:
# Crear DataFrame 'viajes' ordenado 
df = df.sort_values(
    by=['bus_id', 'direction_id', 'date_time'],
    ascending=[True, True, True]
).reset_index(drop=True).copy()

# Calcular deltas de tiempo y distancia en variables locales
delta_tiempo_min = (
    df.groupby(['bus_id', 'direction_id'])['date_time']
    .diff()
    .dt.total_seconds() / 60.0
)
delta_distancia = (
    df.groupby(['bus_id', 'direction_id'])['distancia_en_traza']
    .diff()
)

# Detectar puntos de corte de viaje (pausa > 15m, reinicio de traza o cambio de grupo)
es_nuevo_viaje = (
    (delta_tiempo_min > 15) | 
    (delta_distancia < -1.0) | 
    (delta_tiempo_min.isna())
)

# Generar ID único incremental directo (1, 2, 3... N)
df['trip_id'] = es_nuevo_viaje.cumsum()

total_viajes_no_clean = df['trip_id'].nunique()
print(f"Total de viajes identificados: {total_viajes}")

Total de viajes identificados: 466


### 3.3 Eliminación de viajes desviados

En este modelo se eliminan todos aquellos viajes que terminaron en desvio

In [169]:
# Obtener la lista/serie de trip_ids que tuvieron al menos un desvío
trips_con_desvio = df[df['es_desvio'] == True]['trip_id'].unique()

# Mantener solo los viajes cuyo trip_id NO esté en la lista de desvíos
df = df[~df['trip_id'].isin(trips_con_desvio)].copy()

total_viajes_clean = df['trip_id'].nunique()
print(f"Total de viajes restantes: {total_viajes_clean} -{(total_viajes_clean-total_viajes_no_clean) / total_viajes_no_clean * 100:.2f}% de viajes eliminados")

Total de viajes restantes: 466 --19.10% de viajes eliminados


#### 3.4 Calculo de distancia restante a paradas

En base al GFTS de MBTA se genera un csv con las distancias sobre la traza de cada parada en la ruta (Ambos sentidos)

TODO! Reutilizar el modulo del bot que procesa GFTS. Con ese modulo (vamos a tener que investigar como se relacionan los archivos de la ubicacion de la parada con la ruta, linea etc) usando la misma funcion de calcular distancia sobre traza, obtenemos la distancia de la parada sobre la traza.

Si esos valores le restamos la distancia sobre traza del bondi en este momento -> Nos da la distancia restante! Dato que vamos a añadir al modelo.

Por ultimo el tema de los tiempos de espera:
Se obtiene el momento en el que el bondi llego o paso por la parada (hay que calibrar un margen) -> al hacerlo determinamos el ETA para esa parada.
Una vez que tenemos ese dato de tiempo, hacemos una reconstruccion para atras. Debemos en cada metrica que conforma un viaje restarla al timestampt de llegada, eso nos dara el tiempo que falta para que llegue el colectivo a la parada. NUestro ETA objetivo.

### X Tratamiento de Nulos

### X Eliminación de columnas reduntantes

In [167]:
""" Descomentar al finalizar la limpieza

# Varibles innecesarias
df = df.drop(columns=['date_time',
                      'current_status',
                      'occupancy_status',
                      'bus_id',
                      'distancia_fuera_traza',
                      'es_desvio',
                      'longitude',
                      'latitude',
                      'is_day')
                      
# Varibles eliminadas por no variar en el dataset
df = df.drop(columns=[
                      'route_id',
                      'route_pattern_id',
                      'shape_id',
                      'agency_id',
                      'line_id',
                      'timezone',
                      'road_closure')
                      
# Varibles eliminadas que quizas aporten algo si no las eliminamos
# Se consideran eliminar porque varian muy poco
df = df.drop(columns=[
                      'confidence'
                    ])

"""

" Descomentar al finalizar la limpieza\n\n# Varibles innecesarias\ndf = df.drop(columns=['date_time',\n                      'current_status',\n                      'occupancy_status',\n                      'bus_id',\n                      'distancia_fuera_traza',\n                      'es_desvio',\n                      'longitude',\n                      'latitude',\n                      'is_day')\n\n# Varibles eliminadas por no variar en el dataset\ndf = df.drop(columns=[\n                      'route_id',\n                      'route_pattern_id',\n                      'shape_id',\n                      'agency_id',\n                      'line_id',\n                      'timezone',\n                      'road_closure')\n\n# Varibles eliminadas que quizas aporten algo si no las eliminamos\n# Se consideran eliminar porque varian muy poco\ndf = df.drop(columns=[\n                      'confidence'\n                    ])\n\n"

### X Controles finales